# COMP4318/5318 Assignment 2: Image Classification

**Integrated V3 — Canvas PathMNIST subset, 9-class RGB tissue-patch classification**

### Group number: 163; SID1: 530210074; SID2: 540868894; SID3: 550220550; SID4: 550574604

This executable English notebook preserves the Canvas five-section template, consolidates the team's Gaussian Naive Bayes, MLP, CNN work, and reports two additional non-assessed reference baselines. The training/test split is the **original Canvas split**, not a downloaded public benchmark or Assignment 1 rice data. Original owner notebooks and V2 audit artifacts are preserved unchanged.


This notebook is **an integration of frozen methods**, not a new post-test model-selection exercise. The original source notebooks and V2 Docker execution form the audit trail. We distinguish V1 fixed epoch/parameter choices from V2 validation searches and from the *current* run's final test scores. Never copy V2 numbers into the current-run metric table.

For a fresh-kernel final run, set `V3_RUN_TUNING=0` (default); the eight-trial grids for each required model remain documented and executable by setting `V3_RUN_TUNING=1`, but **no tuning cell is required for final-model execution**. `V3_OUTPUT_DIR` controls generated files. Run with the pinned Docker image for comparability.

For standalone Canvas grading, the notebook also accepts `Assignment2Data/` in the current directory and embeds the **labelled archival** V2 search/metric records as fallback when the reproduction folders are not supplied.


## Setup and dependencies

**Pinned V2 CPU environment:** Python 3.11.11, NumPy 1.26.4, pandas 2.2.3, scikit-learn 1.5.2, SciPy 1.14.1, Matplotlib 3.9.2, TensorFlow 2.17.1, Keras 3.6.0. Container: `linux/amd64`; seed 5318, deterministic TensorFlow CPU settings, restricted threads. Source: `reproduction_v2/requirements-v2.txt`.


In [ ]:
from pathlib import Path
from itertools import product
from time import perf_counter
import hashlib, gc, json, os, platform, sys
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("TF_DETERMINISTIC_OPS", "1")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
os.environ.setdefault("PYTHONHASHSEED", "5318")
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("KERAS_BACKEND", "tensorflow")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.dummy import DummyClassifier
from sklearn.neighbors import NearestCentroid
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                              classification_report, confusion_matrix, ConfusionMatrixDisplay)
import tensorflow as tf
from tensorflow import keras

SEED = 5318
try:
    tf.config.threading.set_intra_op_parallelism_threads(4)
    tf.config.threading.set_inter_op_parallelism_threads(2)
except RuntimeError:
    pass
keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()
RUN_TUNING = os.environ.get("V3_RUN_TUNING", "0").strip() == "1"
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "Canvas Sources" / "Assignment2Data-1" / "Assignment2Data" / "X_train.npy").is_file()), None)
# Submission portability: allow the original Canvas 'Assignment2Data/' adjacent
# to a standalone Notebook, without requiring a checkout of the audit repository.
if ROOT is None:
    ROOT = Path.cwd()
V3_OUTPUT_DIR = Path(os.environ.get("V3_OUTPUT_DIR", str(ROOT / "integration_v3" / "outputs"))).expanduser().resolve()
V3_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MLP_OUTPUT_DIR = V3_OUTPUT_DIR / "mlp"
CNN_OUTPUT_DIR = V3_OUTPUT_DIR / "cnn"
NB_OUTPUT_DIR = V3_OUTPUT_DIR / "naive_bayes"
for output_dir in (MLP_OUTPUT_DIR, CNN_OUTPUT_DIR, NB_OUTPUT_DIR):
    output_dir.mkdir(parents=True, exist_ok=True)
CLASS_NAMES = {0:"Adipose",1:"Background",2:"Debris",3:"Lymphocytes",
               4:"Mucus",5:"Smooth muscle",6:"Normal colon mucosa",
               7:"Cancer-associated stroma",8:"Colorectal adenocarcinoma epithelium"}
MLP_CLASS_NAMES = [CLASS_NAMES[i] for i in range(9)]
CNN_CLASS_NAMES = MLP_CLASS_NAMES.copy()
runtime_environment = dict(python=platform.python_version(), numpy=np.__version__,
    pandas=pd.__version__, scikit_learn=sklearn.__version__,
    tensorflow=tf.__version__, keras=keras.__version__,
    platform=platform.platform(), processor=platform.processor(),
    logical_cpu_count=os.cpu_count(), seed=SEED,
    devices=[d.name for d in tf.config.list_physical_devices()],
    deterministic_ops=True, tuning_performed=RUN_TUNING)
mlp_environment = dict(runtime_environment)
cnn_environment = dict(runtime_environment)
(V3_OUTPUT_DIR / "environment.json").write_text(json.dumps(runtime_environment, indent=2), encoding="utf-8")
print(json.dumps(runtime_environment, indent=2))
print("Outputs:", V3_OUTPUT_DIR)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 150, "font.size": 10})


## 1. Data loading, exploration, and preprocessing

Original Canvas: 32,000 training RGB 28×28 patches and 8,000 held-out test patches, across nine classes. Development is one stratified 80/20 split **inside training only** (fit 25,600; validation 6,400), same indices for all algorithms. Use `float32` fixed `/255` scaling based solely on the training pixel scale. Flatten to 2,352 features for GaussianNB/MLP/baselines; preserve NHWC RGB for CNN. No learned transform or augmentation. Image-level split cannot establish patient/slide-level independence.


In [ ]:
data_candidates = []
if os.environ.get("A2_DATA_DIR"):
    data_candidates.append(Path(os.environ["A2_DATA_DIR"]).expanduser())
data_candidates += [
    ROOT / "Canvas Sources" / "Assignment2Data-1" / "Assignment2Data",
    Path.cwd() / "Assignment2Data",
    Path.cwd().parent / "Assignment2Data",
]
DATA_DIR = next((d for d in data_candidates if (d / "X_train.npy").is_file()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Cannot find Canvas NPY files; place Assignment2Data beside the Notebook or set A2_DATA_DIR.")
X_train = np.load(DATA_DIR / "X_train.npy", mmap_mode="r", allow_pickle=False)
X_test = np.load(DATA_DIR / "X_test.npy", mmap_mode="r", allow_pickle=False)
y_train = np.load(DATA_DIR / "y_train.npy", allow_pickle=False).reshape(-1).astype(int)
y_test = np.load(DATA_DIR / "y_test.npy", allow_pickle=False).reshape(-1).astype(int)
assert X_train.shape == (32000, 28, 28, 3) and X_test.shape == (8000, 28, 28, 3)
assert len(y_train) == len(X_train) and len(y_test) == len(X_test)
class_ids = np.unique(y_train)
assert np.array_equal(class_ids, np.arange(9)) and set(np.unique(y_test)) <= set(class_ids)
assert np.isfinite(X_train).all()
assert 0 <= float(X_train.min()) <= float(X_train.max()) <= 255
pixel_divisor = 255.0 if float(np.max(X_train)) > 1.0 + 1e-6 else 1.0
fit_idx, val_idx = train_test_split(np.arange(len(y_train)),
    test_size=0.20, random_state=SEED, stratify=y_train)
assert len(fit_idx) == 25600 and len(val_idx) == 6400 and set(fit_idx).isdisjoint(set(val_idx))
split_fingerprints = {
    "fit_indices_sha256": hashlib.sha256(fit_idx.astype("<i8").tobytes()).hexdigest(),
    "validation_indices_sha256": hashlib.sha256(val_idx.astype("<i8").tobytes()).hexdigest(),
}
# Verified by V2 integrity audit; reject accidental split changes.
assert split_fingerprints["fit_indices_sha256"] == "191d1629061294ebe7e01fe7a79ac57739fced43795161a0c6b821241311c571"
assert split_fingerprints["validation_indices_sha256"] == "c07d9573b52ee95cb6bd73390fde64f9c2dccd1c2ab14c244a479b1aa0dd3718"
mlp_split_fingerprints = dict(split_fingerprints)
cnn_split_fingerprints = dict(split_fingerprints)
X_fit_img = np.asarray(X_train[fit_idx], dtype=np.float32) / pixel_divisor
X_val_img = np.asarray(X_train[val_idx], dtype=np.float32) / pixel_divisor
y_fit, y_val = y_train[fit_idx], y_train[val_idx]
X_fit, X_val = X_fit_img.reshape(len(y_fit), -1), X_val_img.reshape(len(y_val), -1)
mlp_X_fit, mlp_X_val = X_fit, X_val
MLP_INPUT_DIM = X_fit.shape[1]
MLP_NUM_CLASSES = len(class_ids)
CNN_INPUT_SHAPE = X_fit_img.shape[1:]
CNN_NUM_CLASSES = len(class_ids)
assert MLP_INPUT_DIM == 2352 and CNN_INPUT_SHAPE == (28, 28, 3)
assert X_fit.dtype == np.float32 and X_fit.min() >= 0 and X_fit.max() <= 1.0
print("Data:", X_train.shape, X_test.shape, "fit:", len(fit_idx), "val:", len(val_idx))
print("Fixed pixel divisor:", pixel_divisor)
print("V2-compatible split:", split_fingerprints)
(V3_OUTPUT_DIR / "split_fingerprints.json").write_text(json.dumps(split_fingerprints, indent=2), encoding="utf-8")
train_counts = pd.DataFrame({"class_id":class_ids,
    "class_name":[CLASS_NAMES[int(k)] for k in class_ids],
    "train_count":np.bincount(y_train, minlength=9),
    "fit_count":np.bincount(y_fit, minlength=9),
    "validation_count":np.bincount(y_val, minlength=9)})
display(train_counts)
train_counts.to_csv(V3_OUTPUT_DIR / "train_class_counts.csv", index=False)


### Examples of preprocessed data

Examples are drawn from the **development fit partition** rather than test data. RGB scaling changes numerical range but preserves tissue patch content. Class imbalance is why validation Macro-F1 is the selection metric; the labels are not clinical diagnoses by this notebook.


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(9, 8))
rng = np.random.default_rng(SEED)
for ax, k in zip(axes.flat, class_ids):
    chosen = int(rng.choice(np.flatnonzero(y_fit == k)))
    ax.imshow(X_fit_img[chosen])
    ax.set_title(f"{int(k)} · {CLASS_NAMES[int(k)]}", fontsize=9)
    ax.axis("off")
fig.suptitle("Training-only RGB patches after fixed scaling")
fig.tight_layout()
fig.savefig(V3_OUTPUT_DIR / "training_examples.png", bbox_inches="tight")
plt.show()


## 2. Algorithm design and setup


### Algorithm of choice from first six weeks of course — Gaussian Naive Bayes

Treat the RGB pixels as 2,352 continuous features and estimate independent class-conditional Gaussian densities with class priors. This is a deliberately simple, fast classifier; it ignores spatial correlations among adjacent pixels. The selected tuning dimension is `var_smoothing`.


In [ ]:
# Initial untuned instance. The test set is not used in this section.
initial_start = perf_counter()
initial_model = GaussianNB()
initial_model.fit(X_fit, y_fit)
initial_fit_seconds = perf_counter() - initial_start
initial_start = perf_counter()
initial_predictions = initial_model.predict(X_val)
initial_validation_seconds = perf_counter() - initial_start
print('Initial GaussianNB: fit={:.4f}s, validation={:.4f}s, accuracy={:.4f}, '
      'macro_f1={:.4f}, balanced_accuracy={:.4f}'.format(
          initial_fit_seconds, initial_validation_seconds,
          accuracy_score(y_val, initial_predictions),
          f1_score(y_val, initial_predictions, average='macro'),
          balanced_accuracy_score(y_val, initial_predictions)))


### Fully connected neural network

MLP: `Input(2352) → Dense(h, ReLU) → Dropout(p) → Dense(9, softmax)`; Adam and sparse categorical cross-entropy, 128 samples per batch. Tuning varies width, learning rate and dropout. Original implementation follows.


In [ ]:
MLP_BATCH_SIZE = 128
MLP_MAX_EPOCHS = 12
MLP_PATIENCE = 3

@keras.utils.register_keras_serializable(package='COMP5318')
class MLPMacroF1(keras.metrics.Metric):
    def __init__(self, num_classes=9, name='macro_f1', **kwargs):
        super().__init__(name=name, **kwargs)
        self.num_classes = int(num_classes)
        self.matrix = self.add_weight(name='confusion', shape=(num_classes, num_classes),
                                      initializer='zeros', dtype=tf.float32)

    def update_state(self, y_true, y_pred, sample_weight=None):
        actual = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        predicted = tf.argmax(y_pred, axis=-1, output_type=tf.int32)
        weights = None if sample_weight is None else tf.broadcast_to(
            tf.cast(tf.reshape(sample_weight, [-1]), tf.float32), tf.shape(actual))
        self.matrix.assign_add(tf.math.confusion_matrix(
            actual, predicted, num_classes=self.num_classes, weights=weights, dtype=tf.float32))

    def result(self):
        tp = tf.linalg.diag_part(self.matrix)
        denominator = tf.reduce_sum(self.matrix, axis=0) + tf.reduce_sum(self.matrix, axis=1)
        return tf.reduce_mean(tf.math.divide_no_nan(2.0 * tp, denominator))

    def reset_state(self):
        self.matrix.assign(tf.zeros_like(self.matrix))

    def get_config(self):
        return {**super().get_config(), 'num_classes': self.num_classes}


def mlp_measure(y_true, y_pred):
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'macro_f1': float(f1_score(y_true, y_pred, labels=class_ids,
                                   average='macro', zero_division=0)),
        'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
    }


def mlp_make_dataset(features, targets=None, training=False):
    dataset = tf.data.Dataset.from_tensor_slices(features if targets is None else (features, targets))
    if training:
        dataset = dataset.shuffle(len(features), seed=SEED, reshuffle_each_iteration=True)
    options = tf.data.Options()
    options.experimental_deterministic = True
    options.threading.private_threadpool_size = 2
    options.threading.max_intra_op_parallelism = 1
    return dataset.batch(MLP_BATCH_SIZE).with_options(options).prefetch(1)


def mlp_build_model(hidden_units, learning_rate, dropout_rate):
    keras.backend.clear_session()
    keras.utils.set_random_seed(SEED)
    model = keras.Sequential([
        keras.Input(shape=(MLP_INPUT_DIM,)),
        keras.layers.Dense(hidden_units, activation='relu'),
        keras.layers.Dropout(dropout_rate, seed=SEED),
        keras.layers.Dense(MLP_NUM_CLASSES, activation='softmax'),
    ], name='pathmnist_mlp')
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
                  loss=keras.losses.SparseCategoricalCrossentropy(),
                  metrics=[keras.metrics.SparseCategoricalAccuracy(name='accuracy'),
                           MLPMacroF1(MLP_NUM_CLASSES)], jit_compile=False)
    expected = (MLP_INPUT_DIM + 1) * hidden_units + (hidden_units + 1) * MLP_NUM_CLASSES
    assert model.count_params() == expected
    return model


class MLPEpochLogger(keras.callbacks.Callback):
    def __init__(self, run_name):
        super().__init__()
        self.run_name = run_name
        self.records = []

    def on_epoch_begin(self, epoch, logs=None):
        self._start = perf_counter()
        self._validation_seconds = 0.0

    def on_test_begin(self, logs=None):
        self._validation_start = perf_counter()

    def on_test_end(self, logs=None):
        self._validation_seconds += perf_counter() - self._validation_start

    def on_epoch_end(self, epoch, logs=None):
        elapsed = perf_counter() - self._start
        values = {k: float(v) for k, v in (logs or {}).items()}
        if not all(np.isfinite(v) for v in values.values()):
            raise FloatingPointError(f'Non-finite training metric in {self.run_name}')
        self.records.append({'epoch': epoch + 1, **values, 'epoch_seconds': elapsed,
                             'training_seconds': max(0.0, elapsed - self._validation_seconds),
                             'validation_seconds': self._validation_seconds})
        pd.DataFrame(self.records).to_csv(MLP_OUTPUT_DIR / f'{self.run_name}_history.csv', index=False)


def mlp_fit_development(params, run_name, epochs=MLP_MAX_EPOCHS):
    model = mlp_build_model(**params)
    train_data = mlp_make_dataset(mlp_X_fit, y_fit, training=True)
    validation_data = mlp_make_dataset(mlp_X_val, y_val)
    logger = MLPEpochLogger(run_name)
    early_stop = keras.callbacks.EarlyStopping(monitor='val_macro_f1', mode='max',
        patience=MLP_PATIENCE, min_delta=0.0, restore_best_weights=True)
    start = perf_counter()
    history = model.fit(train_data, validation_data=validation_data, epochs=epochs,
                        callbacks=[logger, early_stop], verbose=2)
    fit_seconds = perf_counter() - start
    prediction_data = mlp_make_dataset(mlp_X_val)
    start = perf_counter()
    probabilities = model.predict(prediction_data, verbose=0)
    validation_seconds = perf_counter() - start
    scores = mlp_measure(y_val, probabilities.argmax(axis=1))
    assert np.isclose(scores['macro_f1'], max(history.history['val_macro_f1']), atol=2e-6)
    result = {
        **params, 'batch_size': MLP_BATCH_SIZE, 'max_epochs': epochs, 'patience': MLP_PATIENCE,
        **{f'validation_{key}': value for key, value in scores.items()},
        'fit_seconds': fit_seconds, 'validation_seconds': validation_seconds,
        'training_epoch_seconds': sum(r['training_seconds'] for r in logger.records),
        'in_fit_validation_seconds': sum(r['validation_seconds'] for r in logger.records),
        'epochs_run': len(logger.records), 'best_epoch': int(np.argmax(history.history['val_macro_f1']) + 1),
        'parameters': int(model.count_params()), 'seed': SEED,
        'fit_samples': len(y_fit), 'validation_samples': len(y_val),
    }
    return model, logger.records, result


# Verify streaming accumulation and reset, including classes absent from batches.
mlp_check_true = np.array([0, 0, 1, 1, 2])
mlp_check_pred = np.array([0, 1, 1, 1, 0])
mlp_check_metric = MLPMacroF1(MLP_NUM_CLASSES)
for mlp_slice in [slice(0, 2), slice(2, None)]:
    mlp_check_metric.update_state(mlp_check_true[mlp_slice],
        tf.one_hot(mlp_check_pred[mlp_slice], MLP_NUM_CLASSES))
assert np.isclose(float(mlp_check_metric.result()),
                  mlp_measure(mlp_check_true, mlp_check_pred)['macro_f1'], atol=1e-6)
mlp_check_metric.reset_state()
assert float(mlp_check_metric.result()) == 0.0
mlp_preview = mlp_build_model(128, 0.001, 0.0)
mlp_preview.summary()
del mlp_preview
print('Macro-F1 streaming/reset and parameter-count checks passed.')


### Convolutional neural network

CNN: two 3×3 convolution/max-pooling blocks (f, 2f), flatten, Dense(64, ReLU), dropout, nine-way softmax. It keeps 28×28 spatial structure. Original architecture, data pipeline, deterministic setup and streaming Macro-F1 are preserved.


In [ ]:
CNN_BATCH_SIZE = 128
CNN_MAX_EPOCHS = 12
CNN_PATIENCE = 3

@keras.utils.register_keras_serializable(package="COMP5318")
class CNNMacroF1(keras.metrics.Metric):
    def __init__(self, num_classes=9, name="macro_f1", **kwargs):
        super().__init__(name=name, **kwargs)
        self.num_classes = int(num_classes)
        self.matrix = self.add_weight(
            name="confusion", shape=(self.num_classes, self.num_classes),
            initializer="zeros", dtype=tf.float32,
        )

    def update_state(self, y_true, y_pred, sample_weight=None):
        actual = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        predicted = tf.argmax(y_pred, axis=-1, output_type=tf.int32)
        weights = None if sample_weight is None else tf.cast(
            tf.reshape(sample_weight, [-1]), tf.float32
        )
        self.matrix.assign_add(tf.math.confusion_matrix(
            actual, predicted, num_classes=self.num_classes,
            weights=weights, dtype=tf.float32,
        ))

    def result(self):
        tp = tf.linalg.diag_part(self.matrix)
        denominator = tf.reduce_sum(self.matrix, axis=0) + tf.reduce_sum(self.matrix, axis=1)
        return tf.reduce_mean(tf.math.divide_no_nan(2.0 * tp, denominator))

    def reset_state(self):
        self.matrix.assign(tf.zeros_like(self.matrix))

    def get_config(self):
        return {**super().get_config(), "num_classes": self.num_classes}


def cnn_measure(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(
            y_true, y_pred, labels=class_ids, average="macro", zero_division=0
        )),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    }


def cnn_make_dataset(images, targets=None, training=False):
    values = images if targets is None else (images, targets)
    dataset = tf.data.Dataset.from_tensor_slices(values)
    if training:
        dataset = dataset.shuffle(len(images), seed=SEED, reshuffle_each_iteration=True)
    options = tf.data.Options()
    options.experimental_deterministic = True
    options.threading.private_threadpool_size = 2
    options.threading.max_intra_op_parallelism = 1
    return dataset.batch(CNN_BATCH_SIZE).with_options(options).prefetch(1)


def cnn_build_model(base_filters, learning_rate, dropout_rate):
    keras.backend.clear_session()
    keras.utils.set_random_seed(SEED)
    model = keras.Sequential([
        keras.Input(shape=CNN_INPUT_SHAPE),
        keras.layers.Conv2D(base_filters, 3, padding="same", activation="relu"),
        keras.layers.MaxPooling2D(pool_size=2),
        keras.layers.Conv2D(2 * base_filters, 3, padding="same", activation="relu"),
        keras.layers.MaxPooling2D(pool_size=2),
        keras.layers.Flatten(),
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dropout(dropout_rate, seed=SEED),
        keras.layers.Dense(CNN_NUM_CLASSES, activation="softmax"),
    ], name="pathmnist_cnn")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
                 CNNMacroF1(CNN_NUM_CLASSES)],
        jit_compile=False,
    )
    return model


class CNNEpochLogger(keras.callbacks.Callback):
    # Save per-epoch history and separate training/validation elapsed times.
    def __init__(self, run_name):
        super().__init__()
        self.run_name = run_name
        self.records = []
        self.training_seconds = 0.0
        self.validation_seconds = 0.0

    def on_epoch_begin(self, epoch, logs=None):
        self._epoch_start = perf_counter()
        self._this_validation_seconds = 0.0

    def on_test_begin(self, logs=None):
        self._validation_start = perf_counter()

    def on_test_end(self, logs=None):
        self._this_validation_seconds += perf_counter() - self._validation_start

    def on_epoch_end(self, epoch, logs=None):
        elapsed = perf_counter() - self._epoch_start
        train_elapsed = max(0.0, elapsed - self._this_validation_seconds)
        self.training_seconds += train_elapsed
        self.validation_seconds += self._this_validation_seconds
        record = {
            "run": self.run_name, "epoch": int(epoch + 1),
            **{k: float(v) for k, v in (logs or {}).items()},
            "epoch_seconds": elapsed, "training_seconds": train_elapsed,
            "validation_seconds": self._this_validation_seconds,
        }
        self.records.append(record)
        pd.DataFrame(self.records).to_csv(
            CNN_OUTPUT_DIR / f"{self.run_name}_history.csv", index=False
        )
        if os.environ.get("CNN_PROGRESS_FILE"):
            with Path(os.environ["CNN_PROGRESS_FILE"]).open("a", encoding="utf-8") as stream:
                stream.write(json.dumps(record) + "\n")


def cnn_fit_development(params, run_name, epochs=CNN_MAX_EPOCHS):
    model = cnn_build_model(**params)
    train_data = cnn_make_dataset(X_fit_img, y_fit, training=True)
    validation_data = cnn_make_dataset(X_val_img, y_val)
    timer = CNNEpochLogger(run_name)
    early_stop = keras.callbacks.EarlyStopping(
        monitor="val_macro_f1", mode="max", patience=CNN_PATIENCE,
        min_delta=0.0, restore_best_weights=True,
    )
    start = perf_counter()
    history = model.fit(
        train_data, validation_data=validation_data, epochs=epochs,
        callbacks=[timer, early_stop, keras.callbacks.TerminateOnNaN()], verbose=2,
    )
    fit_seconds = perf_counter() - start
    start = perf_counter()
    probabilities = model.predict(cnn_make_dataset(X_val_img), verbose=0)
    validation_seconds = perf_counter() - start
    scores = cnn_measure(y_val, np.argmax(probabilities, axis=1))
    best_epoch = int(np.argmax(history.history["val_macro_f1"]) + 1)
    assert np.isclose(
        scores["macro_f1"], max(history.history["val_macro_f1"]), atol=2e-6
    ), "Restored checkpoint / metric mismatch."
    result = {
        **params, "validation_accuracy": scores["accuracy"],
        "validation_macro_f1": scores["macro_f1"],
        "validation_balanced_accuracy": scores["balanced_accuracy"],
        "fit_seconds": fit_seconds,
        "training_epoch_seconds": timer.training_seconds,
        "in_fit_validation_seconds": timer.validation_seconds,
        "validation_seconds": validation_seconds,
        "epochs_run": len(history.history["loss"]), "best_epoch": best_epoch,
        "parameters": int(model.count_params()),
    }
    return model, history, result


# Verify that the streaming metric agrees with the group's metric, including
# classes absent from a small batch. This check needs no test-set predictions.
cnn_check_true = np.array([0, 0, 1, 1, 2])
cnn_check_pred = np.array([0, 1, 1, 1, 0])
cnn_check_metric = CNNMacroF1(CNN_NUM_CLASSES)
cnn_check_metric.update_state(cnn_check_true, tf.one_hot(cnn_check_pred, CNN_NUM_CLASSES))
assert np.isclose(float(cnn_check_metric.result()), f1_score(
    cnn_check_true, cnn_check_pred, labels=class_ids, average="macro", zero_division=0
), atol=1e-6)
print("Macro-F1 consistency check passed.")
cnn_architecture_preview = cnn_build_model(16, 0.001, 0.3)
cnn_architecture_preview.summary()
del cnn_architecture_preview


### Reference-only baselines

Dummy (most frequent) is the class-frequency floor; Nearest Centroid uses Euclidean distance in flattened RGB pixel space. Neither is a required assessed model, and neither replaces the first-six-weeks algorithm.


In [ ]:
def v3_measure(truth, predicted):
    return {"accuracy":float(accuracy_score(truth, predicted)),
            "macro_f1":float(f1_score(truth, predicted,
                       labels=class_ids, average="macro", zero_division=0)),
            "balanced_accuracy":float(balanced_accuracy_score(truth, predicted))}
def v3_baseline(name):
    if name == "Dummy":
        return DummyClassifier(strategy="most_frequent")
    if name == "Nearest Centroid":
        return NearestCentroid(metric="euclidean")
    raise ValueError(name)


## 3. Hyperparameter tuning

**Assessment protocol:** use one common validation split and Macro-F1, not test performance. Each required model has eight parameter trials; MLP/CNN grid-search three independent hyperparameters, with maximum 12 epochs and early-stopping patience 3. Validation checkpoint epochs are selected on validation only.

**Provenance note:** the next tables load *real executed V2 search CSVs* by default. `V3_RUN_TUNING=1` reruns the original tuning code and substitutes new observed search tables, without changing frozen final configurations. The V2 CSV times and scores must not be described as new V3 training.


### Algorithm of choice from first six weeks of course — GaussianNB

Grid: `var_smoothing` ∈ {1e-12, 1e-10, 1e-8, 1e-6, 1e-4, 1e-2, 0.1, 1.0}; a near-flat best plateau supports freezing 1e-6. This tie-breaker was chosen before final test evaluation.


In [ ]:
# Optional controlled re-search from Naive_Bayes source cell 15.
# Default OFF: V2 measured search CSVs remain the declared selection evidence.
if RUN_TUNING:
    # Independent hyperparameter search. Do not use the test set here.
    smoothing_values = [1e-12, 1e-10, 1e-8, 1e-6, 1e-4, 1e-2, 1e-1, 1.0]
    search_records = []
    for smoothing in smoothing_values:
        start = perf_counter()
        model = GaussianNB(var_smoothing=smoothing)
        model.fit(X_fit, y_fit)
        fit_seconds = perf_counter() - start
        start = perf_counter()
        predictions = model.predict(X_val)
        validation_seconds = perf_counter() - start
        search_records.append({
            'var_smoothing': smoothing,
            'validation_accuracy': accuracy_score(y_val, predictions),
            'validation_macro_f1': f1_score(y_val, predictions, average='macro'),
            'validation_balanced_accuracy': balanced_accuracy_score(y_val, predictions),
            'fit_seconds': fit_seconds,
            'validation_seconds': validation_seconds,
        })
    search_results = pd.DataFrame(search_records)
    search_display = search_results.copy()
    search_display['var_smoothing'] = search_display['var_smoothing'].map(lambda value: f'{value:.0e}')
    for column in [
        'validation_accuracy', 'validation_macro_f1', 'validation_balanced_accuracy',
        'fit_seconds', 'validation_seconds'
    ]:
        search_display[column] = search_display[column].map(lambda value: f'{value:.4f}')
    display(search_display)
    pass
else:
    print("Skipped expensive Naive_Bayes search; final parameters remain frozen.")


### Fully connected neural network — MLP

8 trials: hidden units {128,256}, learning rate {0.001,0.0003}, dropout {0,0.3}; observed V2 best config 256/0.001/0.0, V2 best epoch 9. Original final model used the earlier frozen epoch **10** (not V2 best epoch 9); the latter remains preserved for a controlled comparison to V2.


In [ ]:
# Optional controlled re-search from MLP source cell 18.
# Default OFF: V2 measured search CSVs remain the declared selection evidence.
if RUN_TUNING:
    # EXPENSIVE, OPTIONAL: final-model cells do not depend on this search.
    mlp_search_records = []
    mlp_trial_histories = {}
    mlp_search_start = perf_counter()
    for mlp_trial_id, (mlp_width, mlp_lr, mlp_dropout) in enumerate(
            product([128, 256], [0.001, 0.0003], [0.0, 0.3]), start=1):
        mlp_params = {'hidden_units': mlp_width, 'learning_rate': mlp_lr, 'dropout_rate': mlp_dropout}
        print(f'\nTrial {mlp_trial_id}/8: {mlp_params}', flush=True)
        mlp_trial_model, mlp_trial_history, mlp_record = mlp_fit_development(
            mlp_params, f'mlp_trial_{mlp_trial_id:02d}')
        mlp_record['trial_id'] = mlp_trial_id
        mlp_search_records.append(mlp_record)
        mlp_trial_histories[mlp_trial_id] = mlp_trial_history
        pd.DataFrame(mlp_search_records).to_csv(MLP_OUTPUT_DIR / 'mlp_search_results.csv', index=False)
        print('Restored validation Macro-F1:', round(mlp_record['validation_macro_f1'], 5))
        del mlp_trial_model
        gc.collect()
    mlp_search_seconds = perf_counter() - mlp_search_start
    mlp_search_results = pd.DataFrame(mlp_search_records)
    mlp_ranked = mlp_search_results.sort_values(
        ['validation_macro_f1', 'parameters', 'trial_id'], ascending=[False, True, True], kind='stable')
    mlp_selected = mlp_ranked.iloc[0].to_dict()
    mlp_selection_record = {
        'best_params': {'hidden_units': int(mlp_selected['hidden_units']),
                        'learning_rate': mlp_selected['learning_rate'], 'dropout_rate': mlp_selected['dropout_rate']},
        'best_epoch': int(mlp_selected['best_epoch']), 'trial_id': int(mlp_selected['trial_id']),
        'validation_macro_f1': mlp_selected['validation_macro_f1'],
        'validation_accuracy': mlp_selected['validation_accuracy'],
        'validation_balanced_accuracy': mlp_selected['validation_balanced_accuracy'],
        'search_wall_seconds': mlp_search_seconds,
    }
    (MLP_OUTPUT_DIR / 'mlp_selection.json').write_text(json.dumps(mlp_selection_record, indent=2), encoding='utf-8')
    display(mlp_ranked.round(5))
    print('Validation-only selection (freeze before test evaluation):')
    print(json.dumps(mlp_selection_record, indent=2))
    pass
else:
    print("Skipped expensive MLP search; final parameters remain frozen.")


### Convolutional neural network — CNN

8 trials: base filters {8,16}, learning rate {0.001,0.0003}, dropout {0,0.3}; observed V2 best config 16/0.001/0.3, V2 best epoch 12. Original final model used the earlier frozen epoch **11** (not V2 best epoch 12). It is retained to maintain the verified V2 frozen-protocol comparison.


In [ ]:
# Optional controlled re-search from CNN source cell 19.
# Default OFF: V2 measured search CSVs remain the declared selection evidence.
if RUN_TUNING:
    # EXPENSIVE, ISOLATED SEARCH: final-model cells do not depend on this cell.
    cnn_search_records = []
    cnn_search_start = perf_counter()
    for cnn_trial_id, (cnn_filters, cnn_lr, cnn_dropout) in enumerate(
        product([8, 16], [0.001, 0.0003], [0.0, 0.3]), start=1
    ):
        cnn_params = {
            "base_filters": cnn_filters,
            "learning_rate": cnn_lr,
            "dropout_rate": cnn_dropout,
        }
        print(f"\nTrial {cnn_trial_id}/8: {cnn_params}", flush=True)
        cnn_trial_model, cnn_trial_history, cnn_trial_record = cnn_fit_development(
            cnn_params, f"cnn_trial_{cnn_trial_id:02d}"
        )
        cnn_trial_record["trial_id"] = cnn_trial_id
        cnn_search_records.append(cnn_trial_record)
        pd.DataFrame(cnn_search_records).to_csv(
            CNN_OUTPUT_DIR / "cnn_search_results.csv", index=False
        )
        print("Restored validation Macro-F1:", round(cnn_trial_record["validation_macro_f1"], 5))
        del cnn_trial_model, cnn_trial_history
        gc.collect()
    cnn_search_elapsed = perf_counter() - cnn_search_start
    (CNN_OUTPUT_DIR / "cnn_search_timing.json").write_text(
        json.dumps({"search_wall_seconds": cnn_search_elapsed}, indent=2), encoding="utf-8"
    )
    print(f"Complete search wall time: {cnn_search_elapsed:.2f} seconds")
    cnn_search_results = pd.DataFrame(cnn_search_records)
else:
    print("Skipped expensive CNN search; final parameters remain frozen.")


In [ ]:
V2_SEARCH_PATHS = {
    "GaussianNB": ROOT / "reproduction_v2" / "results" / "strict_naive_bayes" / "search_results.csv",
    "MLP": ROOT / "reproduction_v2" / "results" / "strict_mlp" / "search_results.csv",
    "CNN": ROOT / "reproduction_v2" / "results" / "strict_cnn" / "search_results.csv",
}
# Immutable V2 search evidence embedded for standalone Canvas Notebook portability.
# This is NOT a fresh tuning execution and is always labelled historical.
import io
V2_ARCHIVED_SEARCH_CSV = {
  "GaussianNB": "var_smoothing,validation_accuracy,validation_macro_f1,validation_balanced_accuracy,fit_seconds,validation_seconds\n1e-12,0.4040625,0.3749196062459024,0.4188215002921101,0.27050457600012123,0.6418488799999977\n1e-10,0.4040625,0.3749196062459024,0.4188215002921101,0.28571574399984456,0.6210019439999996\n1e-08,0.4040625,0.3749196062459024,0.4188215002921101,0.25299572600010833,0.6369064279999748\n1e-06,0.4040625,0.3749196062459024,0.4188215002921101,0.32157871300000807,0.6447479040000417\n0.0001,0.40390625,0.3747690454914106,0.41863317637497266,0.25187407299995357,0.6363597009999467\n0.01,0.4028125,0.371901044524675,0.41723419710496756,0.2671115020000343,0.7005331770001249\n0.1,0.40265625,0.35973244600692517,0.41607364730989327,0.25013748999981544,0.6194878920000519\n1.0,0.33453125,0.2664152867941853,0.3493539696326224,0.24862661700012723,0.6940182639998511\n",
  "MLP": "hidden_units,learning_rate,dropout_rate,batch_size,max_epochs,patience,validation_accuracy,validation_macro_f1,validation_balanced_accuracy,fit_seconds,validation_seconds,training_epoch_seconds,in_fit_validation_seconds,epochs_run,best_epoch,parameters,seed,fit_samples,validation_samples,trial_id\n128,0.001,0.0,128,12,3,0.428125,0.38453208818536594,0.4177603800462091,6.695859087000031,0.1685243880000371,5.960906126000054,0.6565441010001223,8,5,302345,5318,25600,6400,1\n128,0.001,0.3,128,12,3,0.14953125,0.03406576378264697,0.11396417815818673,6.428917370000022,0.13331010199999582,5.646375421999778,0.7208560899999839,6,3,302345,5318,25600,6400,2\n128,0.0003,0.0,128,12,3,0.44453125,0.41079890679584213,0.4381582671735003,9.502190552000002,0.12424761499994474,8.491496180999661,0.9105742930001952,12,9,302345,5318,25600,6400,3\n128,0.0003,0.3,128,12,3,0.39875,0.32881905836993636,0.37853955001346395,9.952701573000013,0.12497588200005794,8.901736246000155,0.94930339799987,12,11,302345,5318,25600,6400,4\n256,0.001,0.0,128,12,3,0.44546875,0.4288596287516593,0.4438372652524023,15.202407861999973,0.17628685399995447,13.775747518999651,1.313987780000275,12,9,604681,5318,25600,6400,5\n256,0.001,0.3,128,12,3,0.3796875,0.3406959214023795,0.36639945537296464,9.724903538000035,0.14172780800004148,8.890759634999881,0.7657801040002141,7,4,604681,5318,25600,6400,6\n256,0.0003,0.0,128,12,3,0.4315625,0.4065796322296282,0.4438871818333627,13.363099428000055,0.15446483200003058,12.180072425999924,1.0801620760000787,12,10,604681,5318,25600,6400,7\n256,0.0003,0.3,128,12,3,0.416875,0.3578316131119347,0.39949994707173014,12.002625634999958,0.1380959169999869,11.006387825999695,0.9080073050002966,10,7,604681,5318,25600,6400,8\n",
  "CNN": "base_filters,learning_rate,dropout_rate,validation_accuracy,validation_macro_f1,validation_balanced_accuracy,fit_seconds,training_epoch_seconds,in_fit_validation_seconds,validation_seconds,epochs_run,best_epoch,parameters,trial_id\n8,0.001,0.0,0.7203125,0.7084203525938956,0.7125183220933844,32.99492129500004,30.06820421000009,2.8207700200000545,0.37272267100001955,12,9,52217,1\n8,0.001,0.3,0.71140625,0.6931477426046206,0.698460855326819,34.083305300000006,31.041952759999845,2.9326926060001597,0.3886903140000868,12,12,52217,2\n8,0.0003,0.0,0.69046875,0.6758323304039269,0.6796507986625966,35.20658711400006,31.96138008300045,3.132067867999808,0.3897935960000041,12,12,52217,3\n8,0.0003,0.3,0.680625,0.6597537649021756,0.6678345859397191,35.91052649699998,32.7464564899999,3.048226892999992,0.37756421199992474,12,12,52217,4\n16,0.001,0.0,0.730625,0.7245801752640036,0.7219016341916638,48.86839896799995,45.42131078799969,3.3508915810001554,0.4512125429999969,10,7,106089,5\n16,0.001,0.3,0.748125,0.7369693606145916,0.7362167225250756,53.487122888000044,49.52096655299988,3.850076246000185,0.449099724000007,12,12,106089,6\n16,0.0003,0.0,0.719375,0.701526370368286,0.7054982123292443,55.407576318999986,51.4100983449996,3.878063903000225,0.44763024799999585,12,12,106089,7\n16,0.0003,0.3,0.7109375,0.694041785236824,0.698429850372208,54.76312150499996,50.887935363999986,3.758114244000012,0.45147009300001173,12,12,106089,8\n"
}
v2_search_tables = {
    name: pd.read_csv(path) if path.is_file()
          else pd.read_csv(io.StringIO(V2_ARCHIVED_SEARCH_CSV[name]))
    for name, path in V2_SEARCH_PATHS.items()
}
assert all(len(df) == 8 for df in v2_search_tables.values()), "V2 evidence is incomplete."
current_search_tables = (
    {"GaussianNB":search_results, "MLP":mlp_search_results, "CNN":cnn_search_results}
    if RUN_TUNING else v2_search_tables
)
search_provenance = "V3 fresh tuning" if RUN_TUNING else "V2 strictly reproduced, archived"
selection_summary = []
for name, df in current_search_tables.items():
    assert "validation_macro_f1" in df.columns and len(df) == 8
    rows = df.sort_values(["validation_macro_f1"], ascending=False, kind="stable")
    best = rows.iloc[0].to_dict()
    selection_summary.append({"model":name,"source":search_provenance,
                              "best_validation_macro_f1":best["validation_macro_f1"],
                              "best_validation_accuracy":best["validation_accuracy"],
                              "best_epoch": best.get("best_epoch", None),
                              "best_trial":best.get("trial_id", None)})
    print(f"\\n{name}: {search_provenance}")
    display(df.round(5))
selection_table = pd.DataFrame(selection_summary)
display(selection_table)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for ax, (name, frame) in zip(axes, current_search_tables.items()):
    x = np.arange(1, len(frame) + 1)
    ax.bar(x, frame["validation_macro_f1"].to_numpy())
    ax.set(title=f"{name} · validation Macro-F1", xlabel="Search trial",
           ylim=(0, 1), xticks=x)
axes[0].set_ylabel("Macro-F1")
fig.suptitle(f"Search evidence: {search_provenance}")
fig.tight_layout()
fig.savefig(V3_OUTPUT_DIR / "validation_search_comparison.png", bbox_inches="tight")
plt.show()
selection_table.to_csv(V3_OUTPUT_DIR / "validation_selection_summary.csv", index=False)
# Explicit non-interchangeable epoch sources.
epoch_sources = pd.DataFrame([
    {"model":"MLP","V1_frozen_final_epochs":10,"V2_search_best_epoch":int(v2_search_tables["MLP"].loc[4,"best_epoch"])},
    {"model":"CNN","V1_frozen_final_epochs":11,"V2_search_best_epoch":int(v2_search_tables["CNN"].loc[5,"best_epoch"])}
])
display(epoch_sources)
epoch_sources.to_csv(V3_OUTPUT_DIR / "epoch_provenance.csv", index=False)


## 4. Final models

**Independent final retraining:** all 32,000 original training examples; no validation or test observations used for weight updates, checkpoint selection, or further hyperparameter changes. Frozen V1-compatible epochs are carried over to match the V2 strict reproduction. Prediction occurs once per final model in this run. Every comparison below is recomputed from the *current* run's predictions; V2 metrics appear only in a separately labelled historical delta table.


### Algorithm of choice from first six weeks of course — GaussianNB

Frozen `var_smoothing=1e-6`; re-fit on all training vectors. No dependence on earlier search object.


In [ ]:
# This final cell is deliberately independent of the search cell.
BEST_VAR_SMOOTHING = 1e-6
X_train_final = np.asarray(X_train, dtype=np.float32).reshape(len(y_train), -1) / pixel_divisor
X_test_final = np.asarray(X_test, dtype=np.float32).reshape(len(y_test), -1) / pixel_divisor

final_start = perf_counter()
final_model = GaussianNB(var_smoothing=BEST_VAR_SMOOTHING)
final_model.fit(X_train_final, y_train)
final_fit_seconds = perf_counter() - final_start
final_start = perf_counter()
final_predictions = final_model.predict(X_test_final)
final_predict_seconds = perf_counter() - final_start

print('Final GaussianNB parameters:', final_model.get_params())
print(f'Training time: {final_fit_seconds:.4f} seconds')
print(f'Test prediction time: {final_predict_seconds:.4f} seconds')
print(f'Test accuracy: {accuracy_score(y_test, final_predictions):.4f}')
print(f'Test macro F1: {f1_score(y_test, final_predictions, average="macro"):.4f}')
print(f'Test balanced accuracy: {balanced_accuracy_score(y_test, final_predictions):.4f}')
print()
print(classification_report(y_test, final_predictions, digits=4, zero_division=0))
cm = confusion_matrix(y_test, final_predictions, labels=np.arange(len(CLASS_NAMES)))
fig, ax = plt.subplots(figsize=(8, 6))
short_class_names = [
    'Adipose', 'Background', 'Debris', 'Lymphocytes', 'Mucus',
    'Smooth\nmuscle', 'Normal\nmucosa', 'Cancer\nstroma', 'Cancer\nepithelium'
]
ConfusionMatrixDisplay(cm, display_labels=short_class_names).plot(
    ax=ax, cmap='Blues', colorbar=False, values_format='d')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
ax.set_title('GaussianNB test confusion matrix')
plt.tight_layout()
plt.show()


### Fully connected neural network — MLP

Frozen 256 hidden units, lr=0.001, dropout=0, **10** full-data epochs. Training and inference come from the original owner's tested code cells; the search grid is not an execution prerequisite.


In [ ]:
# Freeze the completed validation-only search before first test evaluation.
MLP_BEST_PARAMS = {'hidden_units': 256, 'learning_rate': 0.001, 'dropout_rate': 0.0}
MLP_FINAL_EPOCHS = 10
MLP_SELECTED_VALIDATION = {'trial_id': 5, 'macro_f1': 0.4245453119342078, 'accuracy': 0.46828125, 'balanced_accuracy': 0.4624929838057703}
assert MLP_BEST_PARAMS is not None, 'Freeze actual validation results before final training.'
print('Frozen MLP parameters:', MLP_BEST_PARAMS)
print('Fixed full-data epochs:', MLP_FINAL_EPOCHS)
print('Recorded validation selection:', MLP_SELECTED_VALIDATION)


In [ ]:
mlp_full_train = (np.asarray(X_train, dtype=np.float32) / pixel_divisor).reshape(len(y_train), -1)
mlp_final_data = mlp_make_dataset(mlp_full_train, y_train, training=True)
mlp_final_model = mlp_build_model(**MLP_BEST_PARAMS)
mlp_final_logger = MLPEpochLogger('mlp_final')
mlp_start = perf_counter()
mlp_final_history = mlp_final_model.fit(mlp_final_data, epochs=MLP_FINAL_EPOCHS,
                                     callbacks=[mlp_final_logger], verbose=2)
mlp_final_fit_seconds = perf_counter() - mlp_start
assert len(mlp_final_logger.records) == MLP_FINAL_EPOCHS
print(f'Final fit: {len(y_train)} images, {MLP_FINAL_EPOCHS} epochs, '
      f'{mlp_final_fit_seconds:.3f} s, {mlp_final_model.count_params():,} parameters')
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(np.arange(1, MLP_FINAL_EPOCHS + 1), mlp_final_history.history['loss'], marker='o')
ax.set(title='Independent full-data MLP training (no validation)', xlabel='Epoch', ylabel='Cross-entropy')
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(MLP_OUTPUT_DIR / 'mlp_final_training_curve.png')
plt.show()


In [ ]:
# First/only test prediction call; reuse the cached probabilities below.
mlp_test_features = (np.asarray(X_test, dtype=np.float32) / pixel_divisor).reshape(len(y_test), -1)
mlp_test_data = mlp_make_dataset(mlp_test_features)
mlp_start = perf_counter()
mlp_test_probabilities = mlp_final_model.predict(mlp_test_data, verbose=0)
mlp_test_predict_seconds = perf_counter() - mlp_start
assert mlp_test_probabilities.shape == (len(y_test), MLP_NUM_CLASSES)
assert np.isfinite(mlp_test_probabilities).all()
assert np.allclose(mlp_test_probabilities.sum(axis=1), 1.0, atol=1e-5)
mlp_test_predictions = mlp_test_probabilities.argmax(axis=1)
mlp_test_scores = mlp_measure(y_test, mlp_test_predictions)
mlp_final_result = {
    'model': 'MLP', 'best_hyperparameters': json.dumps(MLP_BEST_PARAMS, sort_keys=True),
    'validation_macro_f1': MLP_SELECTED_VALIDATION['macro_f1'],
    **{f'test_{key}': value for key, value in mlp_test_scores.items()},
    'fit_seconds': mlp_final_fit_seconds, 'prediction_seconds': mlp_test_predict_seconds,
    'parameters': int(mlp_final_model.count_params()), 'epochs': MLP_FINAL_EPOCHS,
    'search_fit_samples': len(y_fit), 'validation_samples': len(y_val),
    'train_samples': len(y_train), 'test_samples': len(y_test), 'seed': SEED,
}
display(pd.DataFrame([mlp_final_result]).T.rename(columns={0: 'value'}))
pd.DataFrame([mlp_final_result]).to_csv(MLP_OUTPUT_DIR / 'mlp_final_results.csv', index=False)
mlp_classification = pd.DataFrame(classification_report(
    y_test, mlp_test_predictions, labels=class_ids, target_names=MLP_CLASS_NAMES,
    output_dict=True, zero_division=0)).T
display(mlp_classification.round(4))
mlp_classification.to_csv(MLP_OUTPUT_DIR / 'mlp_classification_report.csv')
np.save(MLP_OUTPUT_DIR / 'mlp_test_predictions.npy', mlp_test_predictions)
np.save(MLP_OUTPUT_DIR / 'mlp_test_probabilities.npy', mlp_test_probabilities)
(MLP_OUTPUT_DIR / 'mlp_final_summary.json').write_text(json.dumps({
    **mlp_final_result, 'best_hyperparameters': MLP_BEST_PARAMS,
    'environment': mlp_environment, 'split_fingerprints': mlp_split_fingerprints,
}, indent=2), encoding='utf-8')
mlp_final_model.save(MLP_OUTPUT_DIR / 'mlp_final_model.keras')


In [ ]:
mlp_cm = confusion_matrix(y_test, mlp_test_predictions, labels=class_ids)
assert mlp_cm.sum() == len(y_test)
np.savetxt(MLP_OUTPUT_DIR / 'mlp_confusion_matrix.csv', mlp_cm, delimiter=',', fmt='%d')
mlp_cm_normalised = np.divide(mlp_cm, mlp_cm.sum(axis=1, keepdims=True),
    out=np.zeros_like(mlp_cm, dtype=float), where=mlp_cm.sum(axis=1, keepdims=True) != 0)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay(mlp_cm, display_labels=class_ids).plot(
    ax=axes[0], cmap='Blues', colorbar=False, values_format='d')
ConfusionMatrixDisplay(mlp_cm_normalised, display_labels=class_ids).plot(
    ax=axes[1], cmap='Blues', colorbar=False, values_format='.2f')
axes[0].set_title('Frozen MLP: test confusion counts')
axes[1].set_title('Frozen MLP: row-normalised test confusion')
fig.tight_layout()
fig.savefig(MLP_OUTPUT_DIR / 'mlp_confusion_matrices.png')
plt.show()
mlp_pairs = sorted([(int(mlp_cm[i, j]), int(i), int(j))
                    for i in class_ids for j in class_ids if i != j], reverse=True)[:5]
print('Largest directed confusions (count, true class, predicted class):', mlp_pairs)
mlp_per_class = mlp_classification.loc[MLP_CLASS_NAMES]
display(Markdown(
    f"Frozen MLP: **Accuracy {mlp_test_scores['accuracy']:.4f}**, "
    f"**Macro-F1 {mlp_test_scores['macro_f1']:.4f}**, "
    f"**Balanced Accuracy {mlp_test_scores['balanced_accuracy']:.4f}**. "
    f"Highest recall: **{mlp_per_class['recall'].idxmax()}**; "
    f"lowest: **{mlp_per_class['recall'].idxmin()}**. "
    'These diagnostics describe the frozen model, not inputs to further tuning.'))


### Convolutional neural network — CNN

Frozen 16 base filters, lr=0.001, dropout=0.3, **11** full-data epochs. Same original tested architecture and independent fitting.


In [ ]:
# Actual validation-selected choices; no search object is needed.
CNN_BEST_PARAMS = {'base_filters': 16, 'learning_rate': 0.001, 'dropout_rate': 0.3}
CNN_FINAL_EPOCHS = 11
CNN_SELECTED_VALIDATION = {'trial_id': 6, 'macro_f1': 0.7386080336263325, 'accuracy': 0.75125, 'balanced_accuracy': 0.743244847776286}
print('Frozen CNN configuration:', CNN_BEST_PARAMS)
print('Fixed full-data training epochs:', CNN_FINAL_EPOCHS)
print('Recorded selection result:', CNN_SELECTED_VALIDATION)


In [ ]:
# No search object, saved checkpoint or external log is required for this cell.
cnn_full_train_img = np.asarray(X_train, dtype=np.float32) / pixel_divisor
cnn_final_train_data = cnn_make_dataset(cnn_full_train_img, y_train, training=True)
cnn_final_model = cnn_build_model(**CNN_BEST_PARAMS)
cnn_final_logger = CNNEpochLogger("cnn_final")
cnn_final_start = perf_counter()
cnn_final_history = cnn_final_model.fit(
    cnn_final_train_data, epochs=CNN_FINAL_EPOCHS,
    callbacks=[cnn_final_logger, keras.callbacks.TerminateOnNaN()], verbose=2,
)
cnn_final_fit_seconds = perf_counter() - cnn_final_start
print(f"Final training: {len(y_train)} images, {CNN_FINAL_EPOCHS} epochs, "
      f"{cnn_final_fit_seconds:.3f} seconds, {cnn_final_model.count_params():,} parameters")


In [ ]:
# First and only test prediction in this run; all subsequent diagnostics reuse it.
cnn_test_img = np.asarray(X_test, dtype=np.float32) / pixel_divisor
cnn_test_data = cnn_make_dataset(cnn_test_img)
cnn_prediction_start = perf_counter()
cnn_test_probabilities = cnn_final_model.predict(cnn_test_data, verbose=0)
cnn_test_predict_seconds = perf_counter() - cnn_prediction_start
assert cnn_test_probabilities.shape == (len(y_test), CNN_NUM_CLASSES)
assert np.isfinite(cnn_test_probabilities).all()
assert np.allclose(cnn_test_probabilities.sum(axis=1), 1.0, atol=1e-5)
cnn_test_predictions = np.argmax(cnn_test_probabilities, axis=1)
cnn_test_scores = cnn_measure(y_test, cnn_test_predictions)
cnn_final_result = {
    "model": "CNN", "best_hyperparameters": json.dumps(CNN_BEST_PARAMS, sort_keys=True),
    "validation_macro_f1": CNN_SELECTED_VALIDATION["macro_f1"],
    "validation_accuracy": CNN_SELECTED_VALIDATION["accuracy"],
    "test_accuracy": cnn_test_scores["accuracy"],
    "test_macro_f1": cnn_test_scores["macro_f1"],
    "test_balanced_accuracy": cnn_test_scores["balanced_accuracy"],
    "fit_seconds": cnn_final_fit_seconds, "prediction_seconds": cnn_test_predict_seconds,
    "parameters": int(cnn_final_model.count_params()), "epochs": CNN_FINAL_EPOCHS,
    "search_fit_samples": len(y_fit), "validation_samples": len(y_val),
    "train_samples": len(y_train), "test_samples": len(y_test), "seed": SEED,
}
cnn_final_results = pd.DataFrame([cnn_final_result])
display(cnn_final_results.T.rename(columns={0: "value"}))
print(classification_report(
    y_test, cnn_test_predictions, labels=class_ids,
    target_names=CNN_CLASS_NAMES, digits=4, zero_division=0,
))
cnn_classification_report = pd.DataFrame(classification_report(
    y_test, cnn_test_predictions, labels=class_ids, target_names=CNN_CLASS_NAMES,
    output_dict=True, zero_division=0,
)).T
cnn_final_results.to_csv(CNN_OUTPUT_DIR / "cnn_final_results.csv", index=False)
cnn_classification_report.to_csv(CNN_OUTPUT_DIR / "cnn_classification_report.csv")
np.save(CNN_OUTPUT_DIR / "cnn_test_predictions.npy", cnn_test_predictions)
np.save(CNN_OUTPUT_DIR / "cnn_test_probabilities.npy", cnn_test_probabilities)
(CNN_OUTPUT_DIR / "cnn_final_summary.json").write_text(json.dumps({
    **cnn_final_result, "best_hyperparameters": CNN_BEST_PARAMS,
    "environment": cnn_environment, "split_fingerprints": cnn_split_fingerprints,
}, indent=2), encoding="utf-8")
# Optional convenience checkpoint. Final training never requires this file.
cnn_final_model.save(CNN_OUTPUT_DIR / "cnn_final_model.keras")


In [ ]:
cnn_cm = confusion_matrix(y_test, cnn_test_predictions, labels=class_ids)
np.savetxt(CNN_OUTPUT_DIR / "cnn_confusion_matrix.csv", cnn_cm, delimiter=",", fmt="%d")
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay(cnn_cm, display_labels=class_ids).plot(
    ax=axes[0], colorbar=False, cmap="Blues", values_format="d"
)
cnn_cm_normalised = cnn_cm / cnn_cm.sum(axis=1, keepdims=True)
ConfusionMatrixDisplay(cnn_cm_normalised, display_labels=class_ids).plot(
    ax=axes[1], colorbar=False, cmap="Blues", values_format=".2f"
)
axes[0].set_title("Final CNN: test confusion counts")
axes[1].set_title("Final CNN: row-normalised test confusion")
fig.tight_layout()
fig.savefig(CNN_OUTPUT_DIR / "cnn_confusion_matrices.png")
plt.show()

cnn_per_class = pd.DataFrame({
    "class_id": class_ids, "class_name": CNN_CLASS_NAMES,
    "precision": [cnn_classification_report.loc[n, "precision"] for n in CNN_CLASS_NAMES],
    "recall": [cnn_classification_report.loc[n, "recall"] for n in CNN_CLASS_NAMES],
    "f1_score": [cnn_classification_report.loc[n, "f1-score"] for n in CNN_CLASS_NAMES],
    "support": [int(cnn_classification_report.loc[n, "support"]) for n in CNN_CLASS_NAMES],
})
display(cnn_per_class.round(4))
cnn_per_class.to_csv(CNN_OUTPUT_DIR / "cnn_per_class_results.csv", index=False)

# Error examples are post-evaluation diagnostics, never inputs to further tuning.
cnn_wrong_indices = np.flatnonzero(cnn_test_predictions != y_test)
cnn_chosen_errors = np.random.default_rng(SEED).choice(
    cnn_wrong_indices, size=min(9, len(cnn_wrong_indices)), replace=False
)
if len(cnn_chosen_errors):
    fig, axes = plt.subplots(3, 3, figsize=(8, 8))
    for ax in axes.ravel():
        ax.axis("off")
    for ax, index in zip(axes.ravel(), cnn_chosen_errors):
        ax.imshow(cnn_test_img[index])
        ax.set_title(
            f"True {y_test[index]} / Pred {cnn_test_predictions[index]}\n"
            f"Predicted probability {cnn_test_probabilities[index].max():.2f}", fontsize=9
        )
    fig.suptitle("Frozen-model errors: descriptive only; no subsequent tuning", y=1.01)
    fig.tight_layout()
    fig.savefig(CNN_OUTPUT_DIR / "cnn_error_examples.png", bbox_inches="tight")
    plt.show()

cnn_off_diagonal = cnn_cm.copy()
np.fill_diagonal(cnn_off_diagonal, 0)
cnn_error_pairs = sorted(
    [(int(cnn_off_diagonal[i, j]), int(i), int(j))
     for i in class_ids for j in class_ids if i != j], reverse=True
)[:5]
print("Largest directed test confusions (count, true ID, predicted ID):", cnn_error_pairs)


### Auxiliary baseline comparison — Dummy and Nearest Centroid

The baselines are fixed configurations with no selection search; they use the same full-training/test split and pixel scaling as the required models.


In [ ]:
baseline_predictions = {}
baseline_durations = {}
for model_name in ("Dummy", "Nearest Centroid"):
    candidate = v3_baseline(model_name)
    start = perf_counter()
    candidate.fit(X_train_final, y_train)
    fit_seconds = perf_counter() - start
    start = perf_counter()
    predicted = candidate.predict(X_test_final)
    predict_seconds = perf_counter() - start
    baseline_predictions[model_name] = predicted
    baseline_durations[model_name] = (fit_seconds, predict_seconds)
    print(model_name, v3_measure(y_test, predicted))
    del candidate


### Unified independent evaluation, provenance and limitations

Accuracy measures overall predictions, **Macro-F1** weights all nine classes equally, and Balanced Accuracy averages per-class recall. Per-class precision/recall/F1 and confusion matrices help reveal weak performance masked by overall Accuracy. Timing comparisons are descriptive, not hardware-normalized. The V2 archived numbers below are **reference only** and were independently recomputed from saved V2 predictions.


In [ ]:
actual_predictions = {
    "Dummy":baseline_predictions["Dummy"],
    "Nearest Centroid":baseline_predictions["Nearest Centroid"],
    "GaussianNB":final_predictions,
    "MLP":mlp_test_predictions,
    "CNN":cnn_test_predictions,
}
model_info = {
    "Dummy":("fixed most-frequent", *baseline_durations["Dummy"]),
    "Nearest Centroid":("fixed euclidean", *baseline_durations["Nearest Centroid"]),
    "GaussianNB":("var_smoothing=1e-6", final_fit_seconds, final_predict_seconds),
    "MLP":(str(MLP_BEST_PARAMS) + "; epochs=10", mlp_final_fit_seconds, mlp_test_predict_seconds),
    "CNN":(str(CNN_BEST_PARAMS) + "; epochs=11", cnn_final_fit_seconds, cnn_test_predict_seconds),
}
comparison_rows, class_rows = [], []
for name, pred in actual_predictions.items():
    pred = np.asarray(pred).reshape(-1).astype(int)
    assert pred.shape == y_test.shape and np.isin(pred, class_ids).all()
    metrics = v3_measure(y_test, pred)
    description, fit_seconds, predict_seconds = model_info[name]
    comparison_rows.append({"model":name, "required":name in ("GaussianNB","MLP","CNN"),
        "frozen_parameters":description, "source":"V3_CURRENT_RUN",
        **metrics, "fit_seconds":fit_seconds, "prediction_seconds":predict_seconds,
        "train_samples":len(y_train), "test_samples":len(y_test), "seed":SEED})
    report = classification_report(y_test, pred, labels=class_ids,
                                   output_dict=True, zero_division=0)
    for idx in class_ids:
        item = report[str(int(idx))]
        class_rows.append({"model":name,"class_id":int(idx),
            "class_name":CLASS_NAMES[int(idx)],"precision":item["precision"],
            "recall":item["recall"],"f1_score":item["f1-score"],"support":int(item["support"])})
    np.save(V3_OUTPUT_DIR / f"predictions_{name.lower().replace(' ', '_')}.npy", pred)
    cm_now = confusion_matrix(y_test, pred, labels=class_ids)
    pd.DataFrame(cm_now, index=class_ids, columns=class_ids).to_csv(
        V3_OUTPUT_DIR / f"confusion_{name.lower().replace(' ', '_')}.csv")
current_scores = pd.DataFrame(comparison_rows).sort_values("macro_f1",ascending=False)
per_class_scores = pd.DataFrame(class_rows)
display(current_scores.round(5))
current_scores.to_csv(V3_OUTPUT_DIR / "model_comparison.csv", index=False)
per_class_scores.to_csv(V3_OUTPUT_DIR / "per_class_scores.csv", index=False)
fig, ax = plt.subplots(figsize=(10, 4.3))
current_scores.set_index("model")[["accuracy","macro_f1","balanced_accuracy"]].plot(
    kind="bar", ax=ax, ylim=(0,1), rot=0)
ax.set(title="Final performance · V3 current execution",ylabel="Score",xlabel="")
fig.tight_layout()
fig.savefig(V3_OUTPUT_DIR / "model_comparison.png", bbox_inches="tight")
plt.show()
# Read historical V2 results (never substitutes for present execution).
historical_paths = {
    "Dummy": ("strict_baseline","dummy"),
    "Nearest Centroid":("strict_baseline","nearest_centroid"),
    "GaussianNB":("strict_naive_bayes","gaussian_nb"),
    "MLP":("strict_mlp","mlp"),
    "CNN":("strict_cnn","cnn"),
}
# Embedded from the verified V2 metrics (reference only), allowing a standalone
# notebook run when the archived audit directory is not packaged for the grader.
V2_VERIFIED_REFERENCE = {
  "Dummy": {
    "accuracy": 0.14675,
    "macro_f1": 0.02843785577598527,
    "balanced_accuracy": 0.1111111111111111
  },
  "Nearest Centroid": {
    "accuracy": 0.43025,
    "macro_f1": 0.41046943540717495,
    "balanced_accuracy": 0.43251483438248034
  },
  "GaussianNB": {
    "accuracy": 0.4155,
    "macro_f1": 0.38574791151250903,
    "balanced_accuracy": 0.43069610180489404
  },
  "MLP": {
    "accuracy": 0.48925,
    "macro_f1": 0.43867241580923655,
    "balanced_accuracy": 0.4699447730850976
  },
  "CNN": {
    "accuracy": 0.73925,
    "macro_f1": 0.7232551863362164,
    "balanced_accuracy": 0.7265011519342607
  }
}
hist_rows = []
for name,(folder,key) in historical_paths.items():
    evidence_file = ROOT / "reproduction_v2" / "results" / folder / "verified_metrics.json"
    info = (json.loads(evidence_file.read_text(encoding="utf-8"))[key]
            if evidence_file.is_file() else V2_VERIFIED_REFERENCE[name])
    hist_rows.append({"model":name,"V2_accuracy":info["accuracy"],
                      "V2_macro_f1":info["macro_f1"],
                      "V2_balanced_accuracy":info["balanced_accuracy"]})
historical = pd.DataFrame(hist_rows)
v3_vs_v2 = current_scores.merge(historical, on="model", validate="one_to_one")
for metric in ("accuracy","macro_f1","balanced_accuracy"):
    v3_vs_v2[f"V3_minus_V2_{metric}"] = v3_vs_v2[metric] - v3_vs_v2[f"V2_{metric}"]
display(v3_vs_v2[["model","macro_f1","V2_macro_f1","V3_minus_V2_macro_f1"]].round(6))
v3_vs_v2.to_csv(V3_OUTPUT_DIR / "v3_vs_v2_historical.csv", index=False)
(V3_OUTPUT_DIR / "summary.json").write_text(json.dumps({
    "results_source":"V3_CURRENT_RUN (fresh fit and prediction)",
    "historical_comparison_source":"reproduction_v2/results/strict_*/verified_metrics.json",
    "search_evidence":search_provenance,
    "frozen_source":"V1 model definitions/final epochs; reproduced in V2",
    "seed":SEED,"n_train":len(y_train),"n_val":len(y_val),"n_test":len(y_test),
    "scores":comparison_rows,"split_sha256":split_fingerprints,
    "environment":runtime_environment}, indent=2), encoding="utf-8")
print("Independent final metrics and provenance saved to", V3_OUTPUT_DIR)


### Embedded figures (current execution)

The following figures are loaded from PNGs generated by *this run*, rather than from historical V1/V2 Notebook outputs. Explicit embedding makes images visible and preserved even in a headless Docker/Jupyter kernel; these remain descriptive diagnostics of the already frozen models.


In [ ]:
from IPython.display import Image as NotebookImage
# Figures are saved earlier by each original model code cell or V3 comparison.
figure_paths = [
    ("Fit-partition tissue examples", V3_OUTPUT_DIR / "training_examples.png"),
    ("V2 archived or optional fresh validation-search trends", V3_OUTPUT_DIR / "validation_search_comparison.png"),
    ("MLP frozen training", MLP_OUTPUT_DIR / "mlp_final_training_curve.png"),
    ("MLP held-out confusion matrices", MLP_OUTPUT_DIR / "mlp_confusion_matrices.png"),
    ("CNN held-out confusion matrices", CNN_OUTPUT_DIR / "cnn_confusion_matrices.png"),
    ("CNN descriptive error examples", CNN_OUTPUT_DIR / "cnn_error_examples.png"),
    ("Five-model fresh-test comparison", V3_OUTPUT_DIR / "model_comparison.png"),
]
embedded_images = 0
for figure_title, figure_file in figure_paths:
    if figure_file.is_file():
        display(Markdown(f"**{figure_title}**"))
        display(NotebookImage(filename=str(figure_file)))
        embedded_images += 1
    else:
        print("Optional figure not found (diagnostics only):", figure_file)
assert embedded_images >= 3, "The core EDA/search/comparison figures must be embedded."
print(f"Embedded {embedded_images} current-run figures into notebook outputs.")


### Interpretation and limitations

The following narrative is based on the **V2 archived reproduction only**; the table immediately above is the authoritative result for this newly executed notebook. In V2 the CNN achieved Macro-F1 0.723255, outperforming MLP 0.438672 and GaussianNB 0.385748. CNN particularly improves several visually structured classes, but has nontrivial confusion among Debris, Smooth muscle, and Cancer-associated stroma; overall accuracy alone obscures these differences. V1 and V2 MLP/CNN outcomes differ across hardware/OS and dependency versions; the audit does not identify a unique root cause. The Canvas subset is not the full public PathMNIST benchmark, and missing patient/slide identifiers prevent verification of patient-independent splitting. Results are not clinical decision support.

**Do not tune after reading held-out test results.** Any alternative epoch choices (V2 search best 9 and 12) require a clearly marked *new* experiment and must not be silently presented as the original frozen V2 protocol.


## 5. AI Acknowledgement

ChatGPT and Codex (OpenAI) were used across the group workspace for assignment interpretation, code scaffolding and implementation assistance, dependency/Docker design, model reproducibility checking, technical documentation, visualisations and final Notebook integration. Human group members are responsible for checking the generated or assisted code, verifying outputs, interpreting evidence, and declaring their own use of AI in the report. This acknowledgement should be updated with any further AI usage and any required individual reflections **before submission**.

**Source lineage:** `Canvas Sources/COMP5318-assignment2-template-notebook-1.ipynb`; original member notebooks under `archive/v1/notebooks/`; `reproduction_v2/reference/requirements-v2-resolved.txt` and `reproduction_v2/results/strict_*/`; V2 audit reports. The V3 notebook changes *integration and provenance presentation*, not original model architectures or frozen final hyperparameters.
